# 3D Gaussian Splatting — visual reconstruction

**Tanisha Jain · Experiment notebook**

Prepared from the final `Cópia_de_GaussianSplating (1).ipynb`. This workflow trains on the Truck scene, renders reconstructed views, and exports the Gaussian point cloud.


Run this notebook in a **fresh Google Colab GPU runtime**. These are curated experiment
workflows, based on my original testing notebooks. Upstream implementations do the model
training and rendering. CUDA/PyTorch compatibility must be checked against the upstream
environment before installing extensions. The cleaned notebook has been syntax-checked,
but has not been rerun end-to-end on a GPU for this publication.

Sources: [GRAPHDECO authors](https://github.com/graphdeco-inria/gaussian-splatting), [camenduru fork used in the experiment](https://github.com/camenduru/gaussian-splatting).

## 1. Check the GPU and clone the implementation

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
%cd /content/gaussian-splatting
!git rev-parse HEAD

## 2. Build the CUDA extensions
The original experiment needed `<cfloat>` in `simple_knn.cu`. The small compatibility patch below is idempotent. It modifies only the downloaded working copy.

In [ ]:
from pathlib import Path
source = Path("submodules/simple-knn/simple_knn.cu")
text = source.read_text()
if "#include <cfloat>" not in text:
    source.write_text("#include <cfloat>\n" + text)
%pip install plyfile
%pip install ./submodules/diff-gaussian-rasterization
%pip install ./submodules/simple-knn

In [ ]:
import diff_gaussian_rasterization
import simple_knn
print("CUDA extensions imported successfully")

## 3. Prepare the Truck dataset
Use the same T&T/DB archive URL as the original notebook. Review the [original dataset terms](https://www.tanksandtemples.org/) before use. The dataset is downloaded into the runtime and is not included in this repository.

In [ ]:
!wget -nc https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip -q -n tandt_db.zip
SCENE = Path("/content/gaussian-splatting/tandt/truck")
MODEL = Path("/content/gaussian-splatting/output/truck_3dgs")
assert (SCENE / "images").is_dir(), "Check the extracted dataset layout."
assert (SCENE / "sparse").is_dir(), "COLMAP camera/point data are required."

## 4. Train and render
The original completed Truck run used 30,000 iterations. A named output directory replaces the original automatically generated run ID. These commands can take substantial GPU time.

In [ ]:
!python train.py -s {SCENE} -m {MODEL} --iterations 30000
!python render.py -m {MODEL} --iteration 30000

## 5. Compare a training view
This is a qualitative training-view comparison, not held-out test evaluation.

In [ ]:
from IPython.display import display, Image
render_dir = MODEL / "train" / "ours_30000"
rendered = sorted((render_dir / "renders").glob("*.png"))
assert rendered, "Run rendering successfully before comparing images."
name = rendered[0].name
print("Ground truth:")
display(Image(filename=str(render_dir / "gt" / name)))
print("Rendered:")
display(Image(filename=str(render_dir / "renders" / name)))

## 6. Save the final Gaussian point cloud
Mount your own Drive to retain the output after the runtime ends.

In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
point_cloud = MODEL / "point_cloud" / "iteration_30000" / "point_cloud.ply"
assert point_cloud.is_file(), "Training must finish before exporting."
export = Path("/content/drive/MyDrive/gaussian_splatting_results")
export.mkdir(parents=True, exist_ok=True)
shutil.copy2(point_cloud, export / "truck_3dgs_final.ply")
print("Saved:", export / "truck_3dgs_final.ply")

## Experiment notes
The original notebook also contained repeated setup attempts, a separate unfinished custom-scene run, embedded tutorial screenshots, and a public viewer tunnel. Those have been omitted from this focused Truck workflow. Historical training metrics are recorded in [experiment notes](../docs/EXPERIMENTS.md).